# Community District Choropleth

This notebook uses the small demo dataset in `examples/data/` to build a community-district topic map without calling the live Socrata API.

It shows how to:
- load point-capable service-request records
- aggregate complaint topics by community district
- merge those summaries onto boundary geometries
- render a categorical choropleth of dominant topics

In [ ]:
from pathlib import Path
import sys

import nyc311
from IPython.display import display

repo_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "examples").exists() and (candidate / "src").exists()
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from examples.utils import data_path, merge_summary_map, save_choropleth

records = nyc311.load_service_requests(data_path("service_requests_fixture.csv"))
assignments = nyc311.extract_topics(records, nyc311.TopicQuery("Noise - Residential"))
summaries = nyc311.aggregate_by_geography(assignments, geography="community_district")
map_gdf = merge_summary_map(
    summaries,
    boundaries_source=data_path("community_district_boundaries.geojson"),
)
dominant_map = map_gdf[map_gdf["is_dominant_topic"].fillna(False)].copy()

In [ ]:
display(
    dominant_map[
        ["geography_value", "topic", "complaint_count", "share_of_geography"]
    ].sort_values("geography_value")
)

In [ ]:
map_path = save_choropleth(
    dominant_map,
    column="topic",
    title="Dominant noise topics by community district (demo data)",
    filename="community-district-dominant-noise-topics.png",
    cmap="tab20",
    categorical=True,
)

map_path